In [6]:
import sys
sys.path.append("../../")

%load_ext autoreload
%autoreload 2

import pandas as pd

from baselines_finetune import BaseLineTrainer

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Baselines train + eval

This notebook uses `baselines_finetune.py`.

Flow for each model:
1. train best params with `opt_search_*`
2. evaluate on test with `objective_*(eval=True)`

In [2]:
auction_mode = "FPA"  # or "VCG"
best_params_subfolder = f"{auction_mode.lower()}_baseline_n10_rndm_42"

# metric to optimize: CPC_REL / RMSE / SCR
metric = "SCR"
n_trials = 10

In [3]:
data_config = {
    "train": {
        "campaigns_path": f"../../data/{auction_mode.lower()}/campaigns_{auction_mode.lower()}_filtered_train_final.csv",
        "stats_path": f"../../data/{auction_mode.lower()}/stats_{auction_mode.lower()}_filtered_train_final.csv",
    },
    "test": {
        "campaigns_path": f"../../data/{auction_mode.lower()}/campaigns_{auction_mode.lower()}_filtered_test_final.csv",
        "stats_path": f"../../data/{auction_mode.lower()}/stats_{auction_mode.lower()}_filtered_test_final.csv",
    },
}

data_config

{'train': {'campaigns_path': '../../data/fpa/campaigns_fpa_filtered_train_final.csv',
  'stats_path': '../../data/fpa/stats_fpa_filtered_train_final.csv'},
 'test': {'campaigns_path': '../../data/fpa/campaigns_fpa_filtered_test_final.csv',
  'stats_path': '../../data/fpa/stats_fpa_filtered_test_final.csv'}}

In [4]:
trainer = BaseLineTrainer(
    data_config=data_config,
    metric=metric,
    auction_mode=auction_mode,
    base_params_subfolder=best_params_subfolder,
    random_state=42
)


def train_and_eval_model(trainer: BaseLineTrainer, model_name: str, n_trials: int = 30):
    # train on train split and save best params
    getattr(trainer, f"opt_search_{model_name}")(n_trials=n_trials)

    # evaluate on test split using saved params
    eval_value = getattr(trainer, f"objective_{model_name}")(None, eval=True)
    return eval_value

In [5]:
models = ["linear", "tapid", "mpid", "broi"]

results = []
for model_name in models:
    print(f"=== {model_name.upper()} | metric={metric} | auction_mode={auction_mode} ===")
    value = train_and_eval_model(trainer, model_name, n_trials=n_trials)
    results.append({"model": model_name, "metric": metric, "objective_value": value})

results_df = pd.DataFrame(results)
results_df.sort_values("objective_value", ascending=(metric != "SCR"))

[I 2026-02-24 21:55:23,716] A new study created in memory with name: no-name-06b015eb-f2bc-44c8-878e-c482da3da739


=== LINEAR | metric=SCR | auction_mode=FPA ===


[I 2026-02-24 21:57:05,530] Trial 0 finished with value: 31458.397667218815 and parameters: {'coef': 0.01328793865387881, 'lower_clip': 17, 'upper_clip': 8, 'factor': 4.123548312237237}. Best is trial 0 with value: 31458.397667218815.


CPC_REL: 4.0322571653428225, rmse: 2.701533477095753, SCR: 31458.397667218815


[I 2026-02-24 21:59:10,554] Trial 1 finished with value: 26826.655274375742 and parameters: {'coef': 0.0029375693580248803, 'lower_clip': 1, 'upper_clip': 1, 'factor': 7.442442267775767}. Best is trial 0 with value: 31458.397667218815.


...

[W 2026-02-24 23:04:19,234] Trial 8 failed with parameters: {'ro': 0.20349559513080268, 'v_bar': 1.8071456341395584} because of the following error: TypeError('tuple indices must be integers or slices, not str').
Traceback (most recent call last):
  File "/opt/homebrew/lib/python3.11/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/example_notebooks/evaluate_baselines/baselines_finetune.py", line 338, in objective_broi
    return res['score']['MCR']
           ~~~~~~~~~~~~^^^^^^^
TypeError: tuple indices must be integers or slices, not str
[W 2026-02-24 23:04:19,234] Trial 8 failed with value None.


CPC_REL: 1401.9107519602128, rmse: 1.2174532327756844, SCR: 15571.999546977431


[W 2026-02-24 23:05:17,027] Trial 9 failed with parameters: {'ro': 0.7207895500630221, 'v_bar': 0.17888967192999938} because of the following error: TypeError('tuple indices must be integers or slices, not str').
Traceback (most recent call last):
  File "/opt/homebrew/lib/python3.11/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/example_notebooks/evaluate_baselines/baselines_finetune.py", line 338, in objective_broi
    return res['score']['MCR']
           ~~~~~~~~~~~~^^^^^^^
TypeError: tuple indices must be integers or slices, not str
[W 2026-02-24 23:05:17,027] Trial 9 failed with value None.


CPC_REL: 1401.9107519602128, rmse: 1.2174532327756844, SCR: 15571.999546977431
Best trial:


ValueError: Record does not exist.

### Notes

- `objective_*(eval=True)` loads params from:
  `best_params/{best_params_subfolder}/{model}_{metric}_{auction_mode}.pkl`
- This notebook currently runs: `linear`, `tapid`, `mpid`, `mystique`.
- `broi` is not included in this run block.

In [ ]:
models = ["broi"]

# results_v2 = []
for model_name in models:
    print(f"=== {model_name.upper()} | metric={metric} | auction_mode={auction_mode} ===")
    value = train_and_eval_model(trainer, model_name, n_trials=n_trials)
    results.append({"model": model_name, "metric": metric, "objective_value": value})

results_df = pd.DataFrame(results)
results_df.sort_values("objective_value", ascending=(metric != "SCR"))

[I 2026-02-24 23:54:40,600] A new study created in RDB with name: no-name-d93af64e-5050-49a6-89d1-e27ae7f02a62


=== BROI | metric=SCR | auction_mode=FPA ===


[I 2026-02-24 23:55:39,264] Trial 0 finished with value: 15571.999546977431 and parameters: {'ro': 0.40823316659665543, 'v_bar': 122.75826710862329}. Best is trial 0 with value: 15571.999546977431.


CPC_REL: 1401.9107519602128, rmse: 1.2174532327756844, SCR: 15571.999546977431


[I 2026-02-24 23:56:37,925] Trial 1 finished with value: 15571.999546977431 and parameters: {'ro': 14.071095169064515, 'v_bar': 3.7570597138026463}. Best is trial 0 with value: 15571.999546977431.


...

[I 2026-02-25 00:03:17,357] Trial 8 finished with value: 15571.999546977431 and parameters: {'ro': 0.20349559513080268, 'v_bar': 1.8071456341395584}. Best is trial 0 with value: 15571.999546977431.


CPC_REL: 1401.9107519602128, rmse: 1.2174532327756844, SCR: 15571.999546977431


[I 2026-02-25 00:04:14,265] Trial 9 finished with value: 15571.999546977431 and parameters: {'ro': 0.7207895500630221, 'v_bar': 0.17888967192999938}. Best is trial 0 with value: 15571.999546977431.


CPC_REL: 1401.9107519602128, rmse: 1.2174532327756844, SCR: 15571.999546977431
Best trial:
  Value: 15571.999546977431
  Params: 
    ro: 0.40823316659665543
    v_bar: 122.75826710862329
CPC_REL: 1673.199264722307, rmse: 1.2203193763993325, SCR: 16290.10399082903


,model,metric,objective_value
0,linear,SCR,38000.766232
1,tapid,SCR,33976.186431
2,mpid,SCR,31001.231209
3,broi,SCR,16290.103991
